# SDS 06 — Topic-Sensitive PageRank and HITS Reference

Reusable reference notebook for the Scalable Data Science comprehensive exam.

**Design goals**
- beginner-readable reference code;
- pure-Python toy implementations for debugging and hand-checking;
- Spark-native patterns for real exam data;
- explicit dangling-mass handling for personalized PageRank;
- HITS code that preserves zero-score nodes and checks convergence;
- no external Spark packages required.

The supplied SDS exam restricts solutions to built-in Apache Spark libraries, so the Spark sections below use DataFrame/RDD primitives only.


## 1. Problem recognition

- **One general random-surfer importance score:** ordinary PageRank.
- **Bias the random surfer toward a user/topic/seed set:** personalized/topic-sensitive PageRank.
- **Two link roles — good destinations and good directories:** HITS.

Topic-sensitive PageRank changes the teleport vector. HITS is a different algorithm with two mutually reinforcing score vectors.


In [ ]:
import math
from collections import defaultdict


## 2. Pure-Python personalized PageRank — reference implementation

In [ ]:
def personalized_pagerank(nodes, edges, teleport, d=0.85, tol=1e-12, max_iter=200):
    nodes = list(nodes)
    N = len(nodes)
    out = {u: [] for u in nodes}
    for u, v in edges:
        out[u].append(v)

    # Normalize the teleport vector defensively.
    z = sum(float(teleport.get(u, 0.0)) for u in nodes)
    if z <= 0:
        raise ValueError('teleport vector must have positive total mass')
    vprob = {u: float(teleport.get(u, 0.0)) / z for u in nodes}

    rank = {u: 1.0 / N for u in nodes}

    for it in range(1, max_iter + 1):
        incoming = {u: 0.0 for u in nodes}
        dangling = 0.0
        for u in nodes:
            if out[u]:
                share = rank[u] / len(out[u])
                for w in out[u]:
                    incoming[w] += share
            else:
                dangling += rank[u]

        new = {
            u: (1-d) * vprob[u] + d * (incoming[u] + dangling * vprob[u])
            for u in nodes
        }
        residual = sum(abs(new[u] - rank[u]) for u in nodes)
        rank = new
        if residual < tol:
            return rank, it, residual

    return rank, max_iter, residual


In [ ]:
nodes = ['A','B','C','D']
edges = [('A','B'),('A','C'),('B','C'),('C','A'),('C','D')]

ordinary_v = {u: 1.0 for u in nodes}     # normalized internally
cyber_topic_v = {'A': 1.0, 'B': 1.0}    # A/B are the topic seeds

pr_global, it_g, res_g = personalized_pagerank(nodes, edges, ordinary_v)
pr_topic, it_t, res_t = personalized_pagerank(nodes, edges, cyber_topic_v)

print('global:', pr_global, 'sum=', sum(pr_global.values()))
print('topic :', pr_topic,  'sum=', sum(pr_topic.values()))
print('iterations:', it_g, it_t)


### Sanity checks

1. The teleport probabilities must sum to 1 after normalization.
2. The PageRank values should sum to approximately 1.
3. In personalized PageRank, dangling rank is redistributed using the same teleport vector `v`, not automatically uniformly.


## 3. Combining precomputed topic PageRanks

In [ ]:
def combine_topic_ranks(topic_ranks, weights):
    # topic_ranks: dict topic -> dict node -> rank
    # weights: dict topic -> nonnegative weight
    z = sum(weights.values())
    if z <= 0:
        raise ValueError('weights must have positive sum')
    weights = {t: w/z for t, w in weights.items()}
    nodes = set().union(*(r.keys() for r in topic_ranks.values()))
    return {
        u: sum(weights.get(t, 0.0) * topic_ranks[t].get(u, 0.0) for t in topic_ranks)
        for u in nodes
    }


## 4. Spark personalized/topic-sensitive PageRank

In [ ]:
# Run this cell only in a PySpark environment.
from pyspark.sql import functions as F
from pyspark import StorageLevel


In [ ]:
def personalized_pagerank_spark(
    links,                 # DataFrame [src, dst]
    teleport_df,           # DataFrame [id, v], one row per node or subset; will be normalized
    d=0.85,
    tol=1e-8,
    max_iter=50,
    checkpoint_every=5,
):
    # Distinct unweighted graph. Remove this if multiplicity is intentionally meaningful.
    links = links.select('src','dst').dropDuplicates().persist(StorageLevel.MEMORY_AND_DISK)

    nodes = (
        links.select(F.col('src').alias('id'))
             .union(links.select(F.col('dst').alias('id')))
             .distinct()
             .persist(StorageLevel.MEMORY_AND_DISK)
    )
    N = nodes.count()

    outdeg = links.groupBy('src').agg(F.count('*').alias('outdeg')).persist(StorageLevel.MEMORY_AND_DISK)
    weighted_edges = (
        links.join(outdeg, 'src')
             .select('src','dst',(F.lit(1.0)/F.col('outdeg')).alias('p'))
             .persist(StorageLevel.MEMORY_AND_DISK)
    )

    # Build a complete, normalized teleport vector.
    teleport = (
        nodes.join(teleport_df.select('id','v'), 'id', 'left')
             .fillna(0.0, subset=['v'])
    )
    v_sum = teleport.agg(F.sum('v').alias('s')).first()['s']
    if v_sum is None or v_sum <= 0:
        raise ValueError('teleport vector must have positive total mass')
    teleport = teleport.withColumn('v', F.col('v')/F.lit(float(v_sum))).persist(StorageLevel.MEMORY_AND_DISK)

    ranks = nodes.withColumn('rank', F.lit(1.0/N)).persist(StorageLevel.MEMORY_AND_DISK)

    for it in range(1, max_iter+1):
        incoming = (
            weighted_edges.join(ranks.select(F.col('id').alias('src_id'),'rank'),
                                weighted_edges.src == F.col('src_id'))
                          .select(F.col('dst').alias('id'), (F.col('rank')*F.col('p')).alias('c'))
                          .groupBy('id').agg(F.sum('c').alias('incoming'))
        )

        dangling = (
            ranks.join(outdeg, ranks.id == outdeg.src, 'left_anti')
                 .agg(F.sum('rank').alias('d'))
                 .first()['d'] or 0.0
        )

        new = (
            nodes.join(incoming, 'id', 'left')
                 .join(teleport, 'id', 'inner')
                 .fillna(0.0, subset=['incoming'])
                 .withColumn(
                     'rank',
                     (1.0-d)*F.col('v') + d*(F.col('incoming') + F.lit(float(dangling))*F.col('v'))
                 )
                 .select('id','rank')
                 .persist(StorageLevel.MEMORY_AND_DISK)
        )

        residual = (
            ranks.alias('o').join(new.alias('n'),'id')
                 .agg(F.sum(F.abs(F.col('n.rank')-F.col('o.rank'))).alias('r'))
                 .first()['r'] or 0.0
        )
        mass = new.agg(F.sum('rank').alias('s')).first()['s']
        print(f'iter={it:02d} residual={residual:.3e} mass={mass:.12f}')

        ranks.unpersist()
        ranks = new

        if checkpoint_every and it % checkpoint_every == 0:
            ranks = ranks.checkpoint(eager=True)

        if residual < tol:
            break

    return ranks


### Example topic vector in Spark

If `topic_seed_ids` is small, construct a DataFrame where each seed receives `1.0` and all other nodes are filled with zero inside the function. The function normalizes the vector automatically.


In [ ]:
# Example only:
# seeds = [('Malware', 1.0), ('Cybersecurity', 1.0), ('Computer_security', 1.0)]
# teleport_df = spark.createDataFrame(seeds, ['id','v'])
# topic_ranks = personalized_pagerank_spark(links_df, teleport_df)
# topic_ranks.orderBy(F.desc('rank')).show(25, truncate=False)


## 5. Pure-Python HITS — reference implementation

In [ ]:
def hits(nodes, edges, tol=1e-12, max_iter=200):
    nodes = list(nodes)
    out = {u: [] for u in nodes}
    inn = {u: [] for u in nodes}
    for u, v in edges:
        out[u].append(v)
        inn[v].append(u)

    hub = {u: 1.0 for u in nodes}
    auth = {u: 1.0 for u in nodes}

    for it in range(1, max_iter+1):
        new_auth = {v: sum(hub[u] for u in inn[v]) for v in nodes}
        an = math.sqrt(sum(x*x for x in new_auth.values())) or 1.0
        new_auth = {u: x/an for u,x in new_auth.items()}

        new_hub = {u: sum(new_auth[v] for v in out[u]) for u in nodes}
        hn = math.sqrt(sum(x*x for x in new_hub.values())) or 1.0
        new_hub = {u: x/hn for u,x in new_hub.items()}

        residual = (
            sum(abs(new_auth[u]-auth[u]) for u in nodes)
            + sum(abs(new_hub[u]-hub[u]) for u in nodes)
        )
        auth, hub = new_auth, new_hub
        if residual < tol:
            return hub, auth, it, residual

    return hub, auth, max_iter, residual


In [ ]:
nodes2=['H1','H2','A1','A2','A3']
edges2=[('H1','A1'),('H1','A2'),('H2','A2'),('H2','A3')]
hub, auth, it, res = hits(nodes2, edges2)
print('hub:', hub)
print('authority:', auth)
print('iterations:', it, 'residual:', res)


## 6. Spark HITS with convergence and node preservation

In [ ]:
def hits_spark(
    links,                 # DataFrame [src, dst]
    tol=1e-7,
    max_iter=50,
    checkpoint_every=5,
):
    links = links.select('src','dst').dropDuplicates().persist(StorageLevel.MEMORY_AND_DISK)
    nodes = (
        links.select(F.col('src').alias('id'))
             .union(links.select(F.col('dst').alias('id')))
             .distinct()
             .persist(StorageLevel.MEMORY_AND_DISK)
    )

    scores = nodes.withColumn('hub',F.lit(1.0)).withColumn('auth',F.lit(1.0)).persist(StorageLevel.MEMORY_AND_DISK)

    for it in range(1,max_iter+1):
        auth = (
            links.join(scores.select(F.col('id').alias('src_id'),'hub'), links.src==F.col('src_id'))
                 .groupBy('dst').agg(F.sum('hub').alias('auth'))
                 .select(F.col('dst').alias('id'),'auth')
        )
        auth = nodes.join(auth,'id','left').fillna(0.0,subset=['auth'])
        an = auth.agg(F.sqrt(F.sum(F.col('auth')*F.col('auth'))).alias('n')).first()['n'] or 1.0
        auth = auth.withColumn('auth',F.col('auth')/F.lit(float(an)))

        hub = (
            links.join(auth.select(F.col('id').alias('dst_id'),'auth'), links.dst==F.col('dst_id'))
                 .groupBy('src').agg(F.sum('auth').alias('hub'))
                 .select(F.col('src').alias('id'),'hub')
        )
        hub = nodes.join(hub,'id','left').fillna(0.0,subset=['hub'])
        hn = hub.agg(F.sqrt(F.sum(F.col('hub')*F.col('hub'))).alias('n')).first()['n'] or 1.0
        hub = hub.withColumn('hub',F.col('hub')/F.lit(float(hn)))

        new = (
            nodes.join(hub,'id','left').join(auth,'id','left')
                 .fillna(0.0,subset=['hub','auth'])
                 .persist(StorageLevel.MEMORY_AND_DISK)
        )

        residual = (
            scores.alias('o').join(new.alias('n'),'id')
                  .agg((
                      F.sum(F.abs(F.col('n.hub')-F.col('o.hub')))
                      + F.sum(F.abs(F.col('n.auth')-F.col('o.auth')))
                  ).alias('r'))
                  .first()['r'] or 0.0
        )
        print(f'iter={it:02d} residual={residual:.3e}')

        scores.unpersist()
        scores = new

        if checkpoint_every and it % checkpoint_every == 0:
            scores = scores.checkpoint(eager=True)

        if residual < tol:
            break

    return scores


## 7. Exam-style Wikipedia loader

In [ ]:
def load_wikipedia_links(spark, path='pageviews.csv'):
    df = (
        spark.read.option('sep','\t').option('header',False).csv(path)
             .toDF('src','dst','type','count')
    )
    links = (
        df.filter(F.col('type')=='link')
          .select('src','dst')
          .filter(F.col('src').isNotNull() & F.col('dst').isNotNull())
          .dropDuplicates()
    )
    return df, links

# Example:
# df, links = load_wikipedia_links(spark)
# scores = hits_spark(links, tol=1e-6, max_iter=30)
# scores.orderBy(F.desc('hub')).show(25, truncate=False)
# scores.orderBy(F.desc('auth')).show(25, truncate=False)


## 8. Exam critique checklist

### Topic-sensitive PageRank
- Does `v` sum to 1?
- Does dangling mass follow `v`?
- Are edges interpreted correctly and kept distributed?
- Is convergence checked?
- Does total PageRank remain approximately 1?

### HITS
- Authority aggregates **hub scores from incoming edges**.
- Hub aggregates **authority scores over outgoing edges**.
- Normalize both vectors.
- Preserve nodes with legitimate zero scores.
- Prefer tolerance/residual over an unexplained fixed iteration count.
- Keep graph joins/reductions in Spark; collect only scalars and top-k.


## 9. Quick recognition table

| Prompt | Method |
|---|---|
| one overall random-surfer importance | ordinary PageRank |
| bias ranking toward topic/user seeds | personalized/topic-sensitive PageRank |
| combine several topic biases | weighted mixture of topic PageRanks |
| good destinations vs good navigational pages | HITS |
| high-quality node endorsed by strong hubs | authority |
| page that points to strong authorities | hub |
